# Customer Churn Prediction — Feature Engineering (Notebook 2 of 4)

**Goal of this notebook:** Clean the raw data and create new features that will help the model detect churn patterns.
**Why:** Raw data isn't model-ready. `TotalCharges` has bad data types, `customerID` has no predictive value, and grouping raw numbers into buckets/counts often reveals patterns a single column hides.
**Input:** `../data/raw/Telco-Customer-Churn.csv` (saved in notebook 1)
**Output:** `../data/processed/telco_clean.csv`

In [12]:
import pandas as pd
import os

os.makedirs("../data/processed", exist_ok=True)

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.shape

(7043, 21)

In [13]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Fix TotalCharges
Found in EDA: 11 rows have blank strings instead of numbers (new customers with tenure = 0). Convert to numeric and fill blanks with the median.

In [14]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isnull().sum()   # confirm how many became NaN

np.int64(11)

In [15]:
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['TotalCharges'].isnull().sum()   # should be 0 now

C:\Users\Roshi\AppData\Local\Temp\ipykernel_19840\170104857.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


np.int64(0)

### Drop unusable columns
`customerID` is a unique identifier with no predictive value — keeping it would add noise, not signal.

In [16]:
df.drop('customerID', axis=1, inplace=True)
df.shape

(7043, 20)

### Create tenure_bucket
Grouping tenure into ranges makes the "new customer vs long-term customer" pattern (seen clearly in EDA) easier for the model and easier to explain in the dashboard later.

In [23]:
df['tenure_bucket'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12', '13-24', '25-48', '48+']
)

In [25]:
df[df['tenure'] == 0]
df['tenure_bucket'].isnull().sum()

np.int64(0)

In [26]:
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,services_count
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-12,1
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48,2
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-12,2
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,25-48,3
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0-12,0


### Create services_count
Counting how many add-on services (security, backup, tech support, streaming, etc.) a customer has, instead of treating each service as a separate flag. Total engagement often matters more than which specific service.

In [18]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

df['services_count'] = (df[service_cols] == 'Yes').sum(axis=1)
df[['services_count']].value_counts()

services_count
0                 2219
3                 1118
2                 1033
1                  966
4                  852
5                  571
6                  284
Name: count, dtype: int64

### Encode the target variable
Converting Churn from Yes/No text to 1/0 so it can be used directly in modeling and metrics.

In [19]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

### Final check before saving
Confirm no unexpected nulls and that new columns look correct.

In [9]:
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
tenure_bucket       11
services_count       0
dtype: int64

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   object  
 1   SeniorCitizen     7043 non-null   int64   
 2   Partner           7043 non-null   object  
 3   Dependents        7043 non-null   object  
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   object  
 6   MultipleLines     7043 non-null   object  
 7   InternetService   7043 non-null   object  
 8   OnlineSecurity    7043 non-null   object  
 9   OnlineBackup      7043 non-null   object  
 10  DeviceProtection  7043 non-null   object  
 11  TechSupport       7043 non-null   object  
 12  StreamingTV       7043 non-null   object  
 13  StreamingMovies   7043 non-null   object  
 14  Contract          7043 non-null   object  
 15  PaperlessBilling  7043 non-null   object  
 16  PaymentMethod     7043 n

### Save cleaned data
Saving to `data/processed/` keeps this separate from the raw file — notebook 3 (modeling) will load this file, never the raw one.

In [11]:
df.to_csv("../data/processed/telco_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (7043, 22)


## Summary

- Fixed `TotalCharges` (converted to numeric, filled 11 blank values with median)
- Dropped `customerID` (no predictive value)
- Created `tenure_bucket` (0-12, 13-24, 25-48, 48+ months)
- Created `services_count` (count of add-on services per customer)
- Encoded `Churn` as 0/1
- Saved cleaned dataset to `data/processed/telco_clean.csv`

**Next step:** Notebook 03 — build the ML pipeline (ColumnTransformer + SMOTE + classifier), train/test split, hyperparameter tuning.